In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from statsmodels.tsa.arima_model import ARIMA
import pmdarima as pm
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.metrics import r2_score
from scipy import stats
from scipy.special import inv_boxcox

In [2]:
data = pd.read_excel('Forecast_demand_data.xlsx', sheet_name = 'Demand')

In [3]:
data = data.T

# Resetting the index so the first row becomes the header
data.reset_index(inplace=True)
new_header = data.iloc[0]  # Grab the first row for the header
data = data[1:]  # Take the data less the header row
data.columns = new_header  # Set the header row as the df header

# data

In [44]:
def safe_log(x, p):
    """
    Reverses the transformation for a given pandas Series.

    Parameters:
    - x: A pandas Series.
    - p: type of transformation

    Returns:
    - A transformed pandas Series.
    """
    if p == 'log':
        return np.log(x)
    elif p == 'log10':
        return np.log10(x)
    else:
        return np.power(x,p)
    

def safe_return(x, p):
    """
    Reverses the inverse transformation for a given pandas Series.

    Parameters:
    - x: A pandas Series with values that have been log-transformed.
    - p: type of transformation

    Returns:
    - A pandas Series with the original values before the transformation.
    """
    if p == 'log':
        return np.exp(x)
    elif p == 'log10':
        return 10 ** x
    else:
        return np.power(x,1/p)

In [6]:
def remove_leading_zeros(series):
    """
    Removes leading zeros from a pandas Series and returns the trimmed Series.
    """
    # Convert series to boolean where True indicates non-zero values
    non_zero_mask = series != 0
    
    # Find the index of the first non-zero value
    first_non_zero_index = non_zero_mask.idxmax()
    
    # Slice the series from the first non-zero value onwards
    trimmed_series = series.loc[first_non_zero_index:]
    
    return trimmed_series


In [45]:
def process_column_with_forecast(column_data, column_name, p):
    column_data = remove_leading_zeros(column_data).astype(float)
    # column_data = column_data.apply(safe_log)
    # transformed_data, lambda_value = stats.boxcox(column_data)
    column_data = safe_log(column_data, p)
    # print(lambda_value)
    seasonal = False
    model = pm.auto_arima(column_data, 
                          m=1, seasonal=seasonal, d=None, test='adf', 
                          start_p=2, start_q=0, max_p=3, max_q=3, D=None,
                          max_order = 10, information_criterion = 'aic',
                          trace=False, error_action='ignore',  
                          suppress_warnings=True, stepwise=True)
    curr_model = model.order
    model.plot_diagnostics()
    plt.savefig(f'figs/{column_name}_diagnostic_plot.png')
    plt.close()

    # Forecasting
    fc, confint = model.predict(n_periods=10, return_conf_int=True)
    fitted_values = model.predict_in_sample()
    # print(fitted_values)

    # Calculating AIC value
    aic_value = model.aic()

    # Creating a DataFrame for the fitted and forecasted values
    fitted_values = safe_return(fitted_values, p)
    fitted_df = pd.DataFrame(fitted_values)
    fitted_df.rename(columns={'predicted_mean': f'{column_name}_fitted'}, inplace=True)
    # fitted_df.rename(columns={0: f'{column_name}_fitted'}, inplace=True)
    # print(fitted_df)
    
    fc = safe_return(fc, p)
    # fc = inv_boxcox(fc, lambda_value)
    fc_df = pd.DataFrame(fc, columns=[f'{column_name}_forecast'])
    
    # confint = inv_boxcox(confint, lambda_value)
    confint = safe_return(confint,p)
    confidence_df = pd.DataFrame(confint, columns=['Lower', 'Upper'])
    
    #testing changing indices
    if isinstance(data[column].index, pd.DatetimeIndex):
        # If your index is datetime, generate new dates that follow the last date of the historical data
        last_date = data[column].index[-1]
        forecast_dates = pd.date_range(start=last_date, periods=len(fc) + 1, freq='M')[1:]  # Adjust 'freq' as needed
    else:
        # For numeric indices, continue from the last index
        start = data[column].index[-1] + 1
        end = start + len(fc)
        forecast_dates = range(start, end)

    # Assign this new index to fc_df and confidence_df
    fc_df.index = forecast_dates
    confidence_df.index = forecast_dates
    
    print(f"Best Model: ARIMA{model.order}")

    # Adjust the index for the forecasted values and confidence intervals
    # fc_index = pd.RangeIndex(start=len(column_data), stop=len(column_data) + len(fc), step=1)
    # fc_df.index = fc_index
    # confidence_df.index = fc_index
    plt.figure(figsize=(10, 6))
    
    # Plotting historical data
    column_data = safe_return(column_data, p)
    column_data.plot(label='Historical')
    
    # Plotting forecasted values
    plt.plot(fc_df.index, fc_df[f'{column_name}_forecast'], color='red', label='Forecast')
    # Assuming 'fitted_values' is a DataFrame with the same index as 'data'
    # plt.plot(fitted_df.index, fitted_df[f'{column_name}_fitted'], color='green', label='Fitted')
    # Assuming 'confidence_df' is your DataFrame with forecast confidence intervals
    plt.fill_between(confidence_df.index, confidence_df['Lower'], confidence_df['Upper'], color='pink', alpha=0.3)
    plt.legend()
    plt.title(f'ARIMA Forecast with Confidence Intervals for {column_name}')
    plt.xlabel('Time')
    plt.ylabel('Values')
    # plt.text(0.05, 0.95, f'AIC = {aic_value:.3f}', transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', bbox=dict(boxstyle="round", alpha=0.5, facecolor='white'))
    plt.text(0.05, 0.95, f'AIC = {aic_value:.3f}\nModel = {curr_model}', transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', bbox=dict(boxstyle="round", alpha=0.5, facecolor='white'))
    plt.savefig(f'figs/{column_name}_forecast_plot.png')  # Save plot to PNG
    plt.close()

    
    # return fitted_df, fc_df, confidence_df, aic_value
    return confidence_df, aic_value

In [58]:
# Assuming 'data' is your DataFrame with the actual data

# fitted_dfs = {}
# forecast_dfs = {}
confidence_dfs = {}
# AIC_scores = {}
exclude_columns = ['Hib','HPV', 'Pertussis', 'Tetanus', 'Measles', 'Mumps', 'Rubella', 'Polio', 'Hepatitis_B', 'Rotavirus', 'PCV']
df_selected = data.drop(columns=exclude_columns)

for column in df_selected.columns[1:]:
    df_confidence = process_column_with_forecast(data[column], column, 'log')
    # df_fitted, df_forecast, df_confidence, aic_value = process_column_with_forecast(data[column], column)
    # fitted_dfs[column] = df_fitted
    # forecast_dfs[column] = df_forecast
    confidence_dfs[column] = df_confidence
    # AIC_scores[column] = aic_value


Best Model: ARIMA(0, 2, 1)


In [49]:
confidence_dfs = {}
AIC_scores = {}
exclude_columns = ['Hib','HPV', 'Pertussis', 'Tetanus', 'Measles', 'Mumps', 'Rubella', 'Polio', 'Hepatitis_B', 'Rotavirus', 'PCV']
df_selected = data.drop(columns=exclude_columns)

for column in df_selected.columns[1:]:
    for power in np.arange(0.01,0.1, 0.005):
        df_confidence, aic_value = process_column_with_forecast(data[column], column, power)
        confidence_dfs[column] = df_confidence
        AIC_scores[power] = aic_value

Best Model: ARIMA(0, 2, 1)
Best Model: ARIMA(0, 2, 1)
Best Model: ARIMA(0, 2, 1)
Best Model: ARIMA(0, 2, 1)
Best Model: ARIMA(0, 2, 1)
Best Model: ARIMA(0, 2, 1)
Best Model: ARIMA(0, 2, 1)
Best Model: ARIMA(0, 2, 1)
Best Model: ARIMA(0, 2, 1)
Best Model: ARIMA(0, 2, 1)
Best Model: ARIMA(0, 2, 1)
Best Model: ARIMA(0, 2, 1)
Best Model: ARIMA(0, 2, 1)
Best Model: ARIMA(0, 2, 1)
Best Model: ARIMA(0, 2, 1)
Best Model: ARIMA(0, 2, 1)
Best Model: ARIMA(0, 2, 1)
Best Model: ARIMA(0, 2, 1)


In [59]:
confidence_dfs[ 'Diphtheria']

(           Lower         Upper
 25  5.458831e+08  7.770525e+08
 26  5.081606e+08  9.022330e+08
 27  4.726948e+08  1.048355e+09
 28  4.374711e+08  1.224361e+09
 29  4.023732e+08  1.438797e+09
 30  3.677102e+08  1.701738e+09
 31  3.338790e+08  2.025718e+09
 32  3.012560e+08  2.426622e+09
 33  2.701588e+08  2.924747e+09
 34  2.408341e+08  3.546167e+09,
 -75.44301363333025)

In [18]:
excel_path = 'antigen_CIs.xlsx'  # Specify your file path here

with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a') as writer:
    # Iterate over the dictionary and write each DataFrame to a different sheet
    for sheet_name, df in confidence_dfs.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

In [73]:
# Assuming 'data' is your DataFrame with the actual data

# fitted_dfs = {}
# forecast_dfs = {}
confidence_dfs = {}
# AIC_scores = {}
# exclude_columns = ['Diphtheria', 'Pertussis', 'Polio', 'Tetanus']
exclude_columns = ['Hib','HPV']
# df_selected = data.drop(columns=exclude_columns)

for column in data.loc[:, exclude_columns]:
    df_confidence = process_column_with_forecast(data[column], column, p=1)
    # df_fitted, df_forecast, df_confidence, aic_value = process_column_with_forecast(data[column], column)
    # fitted_dfs[column] = df_fitted
    # forecast_dfs[column] = df_forecast
    confidence_dfs[column] = df_confidence
    # AIC_scores[column] = aic_value

Best Model: ARIMA(0, 1, 0)
Best Model: ARIMA(2, 2, 0)


In [74]:
confidence_dfs

{'Hib':            Lower         Upper
 25  2.631497e+08  3.353804e+08
 26  2.602024e+08  3.623521e+08
 27  2.607358e+08  3.858431e+08
 28  2.630709e+08  4.075323e+08
 29  2.665574e+08  4.280702e+08
 30  2.708618e+08  4.477902e+08
 31  2.757860e+08  4.668904e+08
 32  2.812007e+08  4.855000e+08
 33  2.870165e+08  5.037086e+08
 34  2.931680e+08  5.215815e+08,
 'HPV':            Lower         Upper
 25  2.591577e+07  3.196327e+07
 26  3.267239e+07  3.930745e+07
 27  3.525874e+07  4.672036e+07
 28  3.920040e+07  5.298297e+07
 29  4.480188e+07  6.263883e+07
 30  4.746848e+07  6.952104e+07
 31  5.393639e+07  7.989588e+07
 32  5.736408e+07  8.860025e+07
 33  6.325240e+07  9.889957e+07
 34  6.832170e+07  1.095682e+08}